In [1]:
import sys

sys.path.append("..")

In [2]:
from pathlib import Path

from src.inference import Predictor, build_model
from src.preprocessing import (
    ImageProcessor,
TextProcessor,
)
from src.utils.config import load_config
from src.utils.device import get_device

PROJECT_ROOT = Path.cwd().parent

CONFIG_PATH = PROJECT_ROOT / "configs" / "base.yaml"

config = load_config(CONFIG_PATH)

device = get_device(
    config["training"]["device"]
)

print(f"Device: {device}")

c:\Users\AYUSH PRASAD\Desktop\Multimodal Misinformation Detector\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


In [3]:
text_config = config["models"]["text"]
image_config = config["models"]["image"]
fusion_config = config["models"]["fusion"]

text_processor = TextProcessor(
    model_name=text_config["name"],
    max_length=text_config["max_length"],
)

image_processor = ImageProcessor()

model = build_model(
    text_model_name=text_config["name"],
    text_freeze=text_config["freeze"],
    image_freeze=image_config["freeze"],
    hidden_dim=fusion_config["hidden_dim"],
    dropout=fusion_config["dropout"],
    num_classes=config["project"]["num_classes"],
    device=device,
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 24998.83it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
import torch

checkpoint_path = (
    PROJECT_ROOT
    / config["paths"]["checkpoint_dir"]
    / "best_model.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=True,
)

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True,
)

print(
    f"Loaded checkpoint from epoch: "
    f"{checkpoint['epoch']}"
)

Loaded checkpoint from epoch: 3


In [5]:
labels = {
    int(class_id): label_name
    for class_id, label_name
    in config["project"]["labels"].items()
}

predictor = Predictor(
    model=model,
    text_processor=text_processor,
    image_processor=image_processor,
    device=device,
    labels=labels,
)

In [6]:
text = "this reflection on a lake at night"

image_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "images"
    / "test"
    / "d4nf8o.jpg"
)

result = predictor.predict(
    text=text,
    image_path=image_path,
)

result

{'prediction': 'fake',
 'class_id': 1,
 'confidence': 0.5605029463768005,
 'probability_real': 0.4394970238208771,
 'probability_fake': 0.5605029463768005}

In [7]:
print(
    f"Prediction : {result['prediction'].upper()}"
)

print(
    f"Confidence : "
    f"{result['confidence'] * 100:.2f}%"
)

print(
    f"Real       : "
    f"{result['probability_real'] * 100:.2f}%"
)

print(
    f"Fake       : "
    f"{result['probability_fake'] * 100:.2f}%"
)

Prediction : FAKE
Confidence : 56.05%
Real       : 43.95%
Fake       : 56.05%


In [8]:
assert result["prediction"] in {"real", "fake"}

assert 0.0 <= result["confidence"] <= 1.0

assert 0.0 <= result["probability_real"] <= 1.0
assert 0.0 <= result["probability_fake"] <= 1.0

assert (
    abs(
        result["probability_real"]
        + result["probability_fake"]
        - 1.0
    )
    < 1e-5
)

assert (
    abs(
        result["confidence"]
        - max(
            result["probability_real"],
            result["probability_fake"],
        )
    )
    < 1e-5
)

print("All inference sanity checks passed.")

All inference sanity checks passed.
